# 1. 실전 예상문제
제공된 데이터(elec_train.csv)는 타이타닉호 탑승객의 정보를 담고 있다. 이를 바탕으로 생존 여부(Survived)를 예측하는 분류 모델을 개발하고, 가장 우수한 모델을 평가 데이터(titanic_test.csv)에 적용하여 생존 예측 결과를 도출하시오. 예측 결과는 아래의 [제출 형식]을 준수하여, CSV 파일로 생성하는 코드를 제출하시오.
* 예측결과는 ROC-AUC 평가지표에 따라 평가함

[제공 데이터]
1. titanic_train.csv : 학습용 데이터 (891건)
2. titanic_test.csv : 평가용 데이터 (418건, 생존 여부 컬럼 미제공)

[데이터 컬럼]
1. PassengerId : 탑승객 고유 번호
2. Pclass : 선실 등급 (1, 2, 3)
3. Sex : 성별
4. Age : 나이
5. SibSp : 형제자매/배우자 수
6. parch : 부모/자녀 수
7. Fare : 운임
8. Embarked : 탑승항구 (C, Q, S)
9. Survived : 생존 여부 (0: 사망, 1: 생존)

[제출 형식]
1. 제출 파일명 : result.csv
2. 제출 컬럼명 : pred (0, 1)
3. 예측 결과 개수 : 418




In [75]:
# 출력을 원하실 경우 print() 함수 활용
# 예시) print(df.head())

# getcmd(), chdir() 등 작업 폴더 설정 불필요
# 파일 경로 상 내부 드라이버 경로(C: 등) 접근 불가

import pandas as pd

train = pd.read_csv('../../sample_data/part2/이진분류/타이타닉/titanic_train.csv')
test = pd.read_csv('../../sample_data/part2/이진분류/타이타닉/titanic_test.csv')


# 답안 제출 참고
# 아래 코드는 예시이며 변수명 등 개인별로 변경하여 활용
# pd.DataFrame변수.to_csv('result.csv', index=False)

# 여기에 코드를 작성하시오.
X_train = train.drop('Survived', axis=1)
y = train['Survived']

# train, test -> full
full = pd.concat([X_train, test], axis=0)
full = full.drop('PassengerId', axis=1)

# 결측치
full['Age'] = full['Age'].fillna(full['Age'].mean())
full['Fare'] = full['Fare'].fillna(0)
full['Embarked'] = full['Embarked'].fillna(full['Embarked'].mode()[0])

# 범주형 데이터 임베딩
full = pd.get_dummies(full)

# full -> train, test
X_train = full[:train.shape[0]]
X_test = full[train.shape[0]:]

# train, val split
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X_train, y, test_size=0.2, stratify=y)
#print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)

from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier()
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
y_proba = model.predict_proba(X_val)[:,1]

from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
f1 = f1_score(y_val, y_pred)
roc_auc = roc_auc_score(y_val, y_proba)
acc = accuracy_score(y_val, y_pred)
print(f1, roc_auc, acc)

test_pred = model.predict(X_test)
result = pd.DataFrame(test_pred, columns=['pred'])
result
#result.to_csv('result.csv', index=False)

0.7407407407407407 0.8444005270092226 0.8044692737430168


,pred
0,0
1,0
2,0
3,1
4,0
...,...
413,0
414,1
415,0
416,0


# 1.풀이(#1)

In [70]:
import pandas as pd

train = pd.read_csv('../../sample_data/part2/이진분류/타이타닉/titanic_train.csv')
test = pd.read_csv('../../sample_data/part2/이진분류/타이타닉/titanic_test.csv')

# 데이터 유형 파악
#print(train.info())
#print(test.info())
#print(train.head())

# 결측치 파악
#print(train.isnull().sum())
#print(test.isnull().sum())

# 범주형 변수 카테고리 파악
#print(train['Pclass'].value_counts())
#print(train['Sex'].value_counts())
#print(train['Embarked'].value_counts())

# X, Y 데이터 셋 분리
X_train = train.drop(['PassengerId', 'Survived'], axis=1)
y = train['Survived']
X_test = test.drop(['PassengerId'], axis=1)

#print(X_train.shape, y.shape, X_test.shape)

# 결측치 처리
X_train['Age'] = X_train['Age'].fillna(X_train['Age'].mean())
X_train['Embarked'] = X_train['Embarked'].fillna(X_train['Embarked'].mode()[0])

X_test['Age'] = X_test['Age'].fillna(X_test['Age'].mean())
X_test['Fare'] = X_test['Fare'].fillna(0)
#print(X_train.isnull().sum())
#print(X_test.isnull().sum())

# 수치형 변수 스케일링 - MinMaxScaler
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
num_columns = X_train.select_dtypes(exclude='object').columns
X_train[num_columns] = scaler.fit_transform(X_train[num_columns])
X_test[num_columns] = scaler.transform(X_test[num_columns])
#print(X_train.head())

# 범주형 변수 인코딩 - LabelEncoding
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
X_train['Sex'] = encoder.fit_transform(X_train['Sex'])
X_test['Sex'] = encoder.transform(X_test['Sex'])
X_train['Embarked'] = encoder.fit_transform(X_train['Embarked'])
X_test['Embarked'] = encoder.transform(X_test['Embarked'])
#print(X_train['Sex'], X_train['Embarked'])

# 학습, 검증 데이터 셋 분할
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X_train, y, test_size=0.2, stratify=y)
#print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)

# Randomforest 활용 모델 학습
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier()
model.fit(X_train, y_train)

# ROC_AUC, Accuracy 활용 평가
from sklearn.metrics import roc_auc_score, accuracy_score
y_pred = model.predict(X_val)
y_proba = model.predict_proba(X_val)[:, 1]
roc_auc = roc_auc_score(y_val, y_proba)
acc = accuracy_score(y_val, y_pred)
print(roc_auc, acc)

# test 데이터 예측 및 결과 저장 (pred 1개 컬럼)
y_pred = model.predict(X_test)
result = pd.DataFrame(y_pred, columns=['pred'])
result.to_csv('result.csv', index=False)

0.8666007905138341 0.8044692737430168


# 1.풀이(#2)

In [76]:
import pandas as pd

train = pd.read_csv('../../sample_data/part2/이진분류/타이타닉/titanic_train.csv')
test = pd.read_csv('../../sample_data/part2/이진분류/타이타닉/titanic_test.csv')

# 데이터 유형 파악
#print(train.info())
#print(test.info())
#print(train.head())

# 결측치 파악
#print(train.isnull().sum())
#print(test.isnull().sum())

# 범주형 변수 카테고리 파악
#print(train['Pclass'].value_counts())
#print(train['Sex'].value_counts())
#print(train['Embarked'].value_counts())

# X, Y 데이터 셋 분리
X = train.drop(['Survived'], axis=1)
y = train['Survived']

X_full = pd.concat([X, test], axis=0)
X_full = X_full.drop(['PassengerId'], axis=1)
#print(X_full.shape)

# 결측치 처리
X_full['Age'] = X_full['Age'].fillna(X_full['Age'].mean())
X_full['Embarked'] = X_full['Embarked'].fillna(X_full['Embarked'].mode()[0])
X_full['Fare'] = X_full['Fare'].fillna(0)

#print(X_full.isnull().sum())

# 수치형 변수 스케일링 - MinMaxScaling
# Skip!!

# 범주형 변수 인코딩 - One-Hot Encoding
X_full = pd.get_dummies(X_full)
#print(X_full)

# 학습, 검증 데이터 셋 분할
X_train = X_full[:train.shape[0]]
X_test = X_full[train.shape[0]:]
#print(X_train.shape, X_test.shape)

from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X_train, y, test_size=0.2, stratify=y)
#print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)

# Randomforest 활용 모델 학습
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier()
model.fit(X_train, y_train)

# ROC_AUC, Accuracy 활용 평가
from sklearn.metrics import roc_auc_score, accuracy_score
y_pred = model.predict(X_val)
y_proba = model.predict_proba(X_val)[:, 1]
roc_auc = roc_auc_score(y_val, y_proba)
acc = accuracy_score(y_val, y_pred)
print(roc_auc, acc)

# test 데이터 예측 및 결과 저장 (pred 1개 컬럼)
y_pred = model.predict(X_test)
result = pd.DataFrame(y_pred, columns=['pred'])
result
#result.to_csv('result.csv', index=False)

0.8376152832674572 0.7821229050279329


,pred
0,0
1,0
2,0
3,1
4,1
...,...
413,0
414,1
415,0
416,0


# 2. 실전 예상문제
제공된 학습용 데이터(car_insur_train.csv)는 차량 소유자의 연령, 성별, 차량 정보 및 보험 여부 등을 포함한 데이터이다. 해당 데이터를 기반으로 보험 가입 여부를 예측하는 분류 모델을 개발하고, 가장 우수한 모델을 평가 데이터(car_insur_test.csv)에 적용하여 생존 예측하시오. 예측 결과는 아래의 [제출 형식]을 준수하여, CSV 파일로 생성하는 코드를 제출하시오.
* 예측결과는 Accuracy 평가지표에 따라 평가함

[제공 데이터]
1. car_insur_train.csv : 학습용 데이터 (3,811건)
2. car_insur_test.csv : 평가용 데이터 (1,270건, 가입 여부 컬럼 미제공)

[데이터 컬럼]
1. id : 고유 식별자
2. Gender : 성별 (Male, Female)
3. Age : 나이
4. Driving_License : 운전면허 보유 여부 (1: 있음, 0: 없음)
5. Region_Code : 지역 코드
6. Previously_Insured : 이전 보험 가입 여부 (1: 있음, 0: 없음)
7. Vehicle_age : 차량 연식 ( < 1 Year, 1-2 Year, > 2 Years)
8. Vehicl_Damage : 과거 차량 손상 이력
9. ... 기타 여러가지 특성
10. Response : 보험 가입 여부 (1: 가입, 0: 미가입)

[제출 형식]
1. 제출 파일명 : result.csv
2. 제출 컬럼명 : pred (0, 1)
3. 예측 결과 개수 : 1,270




In [142]:
# 출력을 원하실 경우 print() 함수 활용
# 예시) print(df.head())

# getcmd(), chdir() 등 작업 폴더 설정 불필요
# 파일 경로 상 내부 드라이버 경로(C: 등) 접근 불가

import pandas as pd

train = pd.read_csv('../../sample_data/part2/이진분류/자동차보험/car_insur_train.csv')
test = pd.read_csv('../../sample_data/part2/이진분류/자동차보험/car_insur_test.csv')

# 답안 제출 참고
# 아래 코드는 예시이며 변수명 등 개인별로 변경하여 활용
# pd.DataFrame변수.to_csv('result.csv', index=False)

# 여기에 코드를 작성하시오.
# train 독립변수, 종속 변수 나누기
X_train = train.drop('Response', axis=1)
y = train['Response']

# X_train + test = full
X_full = pd.concat([X_train, test], axis=0)

# 결측치 처리 -> 결측치 없음.
# 필요 없는 컬럼 삭제
X_full = X_full.drop('id', axis=1)
#X_full.info() #Gender, Vehicle_age, Vehicle_Damage

# One-hot 인코딩
X_full = pd.get_dummies(X_full)

# full -> X_train, X_test
X_train = X_full[:train.shape[0]]
X_test = X_full[train.shape[0]:]

# 학습 모델 분리
import sklearn
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X_train, y, test_size=0.2, stratify=y)

# 학습하기
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier()
model.fit(X_train, y_train)

# 검증하기
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score, recall_score
y_pred = model.predict(X_val)
y_proba = model.predict_proba(X_val)[:,1]

roc_auc = roc_auc_score(y_val, y_proba)
f1 = f1_score(y_val, y_pred)
acc = accuracy_score(y_val, y_pred)
print(roc_auc, acc, f1)

# 결과 출력
test_pred = model.predict(X_test)
result = pd.DataFrame(test_pred, columns=["pred"])
result

0.8572701011073665 0.872870249017038 0.17094017094017094


,pred
0,0
1,0
2,0
3,0
4,0
...,...
1265,0
1266,0
1267,0
1268,0


# 2.풀이

In [146]:
import pandas as pd

train = pd.read_csv('../../sample_data/part2/이진분류/자동차보험/car_insur_train.csv')
test = pd.read_csv('../../sample_data/part2/이진분류/자동차보험/car_insur_test.csv')

# 데이터 유형 파악
#print(train.info())
#print(test.info())

# 결측치 파악
#print(train.isnull().sum())
#print(test.isnull().sum())

# 범주형 변수 카테고리 파악
#print(train['Gender'].value_counts())
#print(train['Region_Code'].value_counts())
#print(train['Vehicle_Age'].value_counts())
#print(train['Vehicle_Damage'].value_counts())

# X, Y 데이터 셋 분리
X = train.drop(['Response'], axis=1)
y = train['Response']

X_full = pd.concat([X, test], axis=0)
X_full = X_full.drop(['id'], axis=1)
#print(X_full.shape)

# 범주형 변수 인코딩 - One-Hot Encoding
X_full = pd.get_dummies(X_full)
#print(X_full)

# 학습, 검증 데이터 셋 분할
X_train = X_full[:train.shape[0]]
X_test = X_full[train.shape[0]:]
#print(X_train.shape, X_test.shape)

from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X_train, y, test_size=0.2, stratify=y)
#print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)

# Randomforest 활용 모델 학습
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier()
model.fit(X_train, y_train)

# ROC_AUC, Accuracy 활용 평가
from sklearn.metrics import roc_auc_score, accuracy_score
y_pred = model.predict(X_val)
y_proba = model.predict_proba(X_val)[:,1]
acc = accuracy_score(y_val, y_pred)
roc_auc = roc_auc_score(y_val, y_proba)
print(acc,roc_auc)

# test 데이터 예측 및 결과 저장 (pred 1개 컬럼)
y_pred = model.predict(X_test)
result = pd.DataFrame(y_pred, columns=['pred'])
result
#result.to_csv('result.csv', index=False)

0.8623853211009175 0.8242577435403626


,pred
0,0
1,0
2,0
3,0
4,0
...,...
1265,0
1266,0
1267,0
1268,0
